# Exp 029 — inline CoT user-state extraction (DEVSET)

**Single-axis change vs 021 champion**: response prompt switches from stock `response_generation.txt` to `response_generation_cot_user_state.txt`. The LM emits a strict structured `<user_state>` block (mood / intent / energy / sonic_pref / era_pref / familiarity) followed by `<response>...</response>`. `crs_baseline.extract_cot_response` strips the user_state envelope so `predicted_response` going to Gemini is the response prose only.

**Why DEVSET first**: per fresh-model policy, no Blind-A ship without explicit gate. This notebook validates (a) Qwen 1.5B follows the structured format end-to-end on 8000 rows, (b) the parser empty-rate is low, (c) qualitative response read confirms the user_state grounding helps prose specificity. Only after that do we consider a Blind-A ship.

**What to look at after the run**:
- Sample 10 random `predicted_response` values (cell 8). They should be 2-3 sentences, NO `<user_state>` text leaking, NO field names like `mood:` in the prose.
- Parser leak rates (cell 8). High (>5%) = the parser missed cases — file an update to extract_cot_response.
- `local_eval.py` retrieval composite should be ~equal to 020/021 (retrieval stack unchanged); LLM-judge is dev-side untracked but qualitative read informs Blind-A risk.

Wall time: ~7-10 min on A100 (max_new_tokens=192, ~3x the 021 budget).

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone — pull latest fresh-model code.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 3) Install deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters.
TID = '029-cot-user-state-qwen15b-devset'
BATCH_SIZE = 32
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run devset two-step inference.
# response_max_new_tokens=192 (set in yaml) to fit user_state block + response.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_devset.py \
    --tid {TID} \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate prediction JSON + zip for download.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/devset/{TID}.json'
assert os.path.isfile(SRC), f'prediction not found at {SRC} — did inference fail?'

with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 8000)')
assert len(rows) >= 8000, f'only {len(rows)} rows — partial run; do not score'

stage = f'/content/_stage_{TID}'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, f'{TID}.json'))
zip_base = f'/content/{TID}'
shutil.make_archive(zip_base, 'zip', stage)
print('wrote', zip_base + '.zip')
!ls -lh {zip_base}.zip

In [ ]:
# 7a) Browser download.
from google.colab import files
files.download(f'/content/{TID}.zip')

In [ ]:
# 7b) Drive backup.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst_dir = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(f'/content/{TID}.zip', dst_dir)
shutil.copy(f'music-crs-baselines/exp/inference/devset/{TID}.json', dst_dir)
print(f'saved to Drive: {dst_dir}')
!ls -lh {dst_dir}

In [ ]:
# 8) Quality probe — sample responses + parser-leak rate.
# The CoT post-processor in crs_baseline.extract_cot_response prefers
# <response>...</response>; if missing, it strips <user_state>...</user_state>
# and returns the rest. The FINAL parsed text going to Gemini should be
# near-zero leak (no <user_state> / <response> tags, no field names like 'mood:').
import json, random, re

with open(f'music-crs-baselines/exp/inference/devset/{TID}.json') as f:
    rows = json.load(f)

field_leak_re = re.compile(
    r'^(?:mood|intent|energy|sonic_pref|era_pref|familiarity):',
    re.M,
)
tag_leak_re = re.compile(r'<\s*/?\s*(user_state|response)\s*>', re.I)

leak_field, leak_tag, empty = 0, 0, 0
for r in rows:
    resp = (r.get('predicted_response') or '').strip()
    if not resp:
        empty += 1
        continue
    if field_leak_re.search(resp):
        leak_field += 1
    if tag_leak_re.search(resp):
        leak_tag += 1

n = len(rows)
print(f'rows total            : {n}')
print(f'empty responses       : {empty}  ({empty/n:.1%})')
print(f'field-name leak       : {leak_field}  ({leak_field/n:.1%})')
print(f'tag leak              : {leak_tag}  ({leak_tag/n:.1%})')
print()
print('=== 10 random sample responses ===')
random.seed(42)
for i in random.sample(range(n), min(10, n)):
    print(f'\n[{i}] turn {rows[i].get("turn_number")}')
    print(rows[i].get('predicted_response', '')[:400])

# Heuristic gate: tag/field-leak rate >5% in the FINAL parsed response
# means parser failure (unexpected — local smoke had 0% leak). It is fine
# if the LM only emits the structured <user_state> block ~70% of the time
# (1.5B drops the format on some queries — local smoke showed ~67%
# follow-rate). The parser falls back cleanly when the format is missing.

## After the Colab run, on local M4:

```bash
cd recsys2026
TID=029-cot-user-state-qwen15b-devset
unzip -o ~/Downloads/${TID}.zip -d music-crs-baselines/exp/inference/devset/
source recsys26/bin/activate
python scripts/local_eval.py --tid ${TID} --split dev
pytest tests/test_wave2_integration.py -v
```

## Decision gate

- **Empty/leak rate <5% AND retrieval composite within ±0.005 of 021 dev** → eligible for Blind-A consideration; log to `documents/experiments_log.md` and re-rank `project_next_experiment_candidates.md`.
- **Empty/leak rate ≥5%** → 1.5B isn't following the strict format. Iterate prompt (simpler/fewer fields, or move to a 2-call pipeline) before Blind-A.
- **Retrieval composite drops** → shouldn't happen since retrieval stack is unchanged; investigate before any further runs.